In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


In [0]:
%run /Workspace/consolidated_pipeline/consolidate_pipeline/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','orders','Data Source')

catalog=dbutils.widgets.get('catalog')
data_source=dbutils.widgets.get('data_source')

base_path=f's3://sportsbar-pro/{data_source}'
landing_path=f"{base_path}/landing/"
processed_path=f'{base_path}/processsed/'
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)


bronze_table=f"{catalog}.{bronze_schema}.{data_source}"
silver_table=f"{catalog}.{silver_schema}.{data_source}"
gold_table=f"{catalog}.{gold_schema}.sb_fact{data_source}"


print("Bronze Table: ", bronze_table)
print("Silver Table: ", silver_table)
print("Gold Table: ", gold_table)

In [0]:
bronze_table,silver_table,gold_table

### Bronze

In [0]:
files=[
    f for f in dbutils.fs.ls(landing_path)
    if f.name.lower().endswith('.csv')
]

if not files:
    print('No New csv files found in landing. Nothing to process.')
    dbutils.notebook.exit('No_NEW_FILES')

print('New Files Found:')
for file_info in files:
    print(file_info.name)

In [0]:
# Check for new CSV files in the landing folder

files = [
    f for f in dbutils.fs.ls(landing_path)
    if f.name.lower().endswith(".csv")
]

if not files:
    print("No new CSV files found in landing folder.")
    print("Nothing to process.")
    dbutils.notebook.exit("NO_NEW_FILES")

print("Files found in landing folder:")
for file_info in files:
    print(file_info.name)

In [0]:
df = spark.read.options(
    header=True,
    inferSchema=True
).csv(
    f"{landing_path}/*.csv"
).withColumn(
    "read_timestamp",
    F.current_timestamp()
).select(
    "*",
    "_metadata.file_name",
    "_metadata.file_size"
)

print("Total Rows: ", df.count())
df.show(5)

In [0]:
# (
#     df.write
#     .format('delta')
#     .option('delta.enableChangeDataFeed','true')
#     .mode('append')
#     .saveAsTable(bronze_table)
# )


if not spark.catalog.tableExists(bronze_table):
    print(f"Bronze table does not exist. Creating {bronze_table}")

    (
        df.write
        .format('delta')
        .option('delta.enableChangeDataFeed','true')
        .mode('overwrite')
        .saveAsTable(bronze_table)
    )


else:
    print(f"Bronze table {bronze_table} already exists.")

    bronze_delta=DeltaTable.forName(spark,bronze_table)

    bronze_delta.alias('target').merge(
        df.alias('source'),
        """
        target.order_id = source.order_id
        AND target.order_placement_date = source.order_placement_date
        AND target.customer_id = source.customer_id
        AND target.product_id = source.product_id
        AND target.order_qty = source.order_qty
        """
    ).whenNotMatchedInsertAll().execute()

    print('Bronze incremental merge completed')

### Staging table to process just the arrived incremental data

In [0]:
(
    df.write
    .format('delta')
    .option('delta.enableChangeDataFeed','true')
    .mode('overwrite')
    .saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")
)

In [0]:
df.show()

### Moving files from source to processed directory

In [0]:
files=dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(file_info.path,f"{processed_path}/{file_info.name}",
    True
    )

In [0]:
df.columns

### Silver

In [0]:
df_orders= spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.staging_{data_source}")
df_orders.show()

##### Transfromations

In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
df_orders.show(1000)

In [0]:
df_orders.agg(
    F.min('order_placement_date').alias('min_date'),
    F.max('order_placement_date').alias('max_date')
).show()

#### Join with products

In [0]:
df_products=spark.table('fmcg.silver.products')
df_joined = df_orders.join(
    df_products,
    on='product_id',
    how='inner'
).select(df_orders['*'],df_products['product_code'])

df_joined.show()

In [0]:
if not(spark.catalog.tableExists(silver_table)):
    (
        df_joined.write
        .format('delta')
        .option('delta.enableChangeDataFeed','true')
        .option('mergeSchema','true')
        .mode('overwrite')
        .saveAsTable(silver_table)
    )

else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
